# 1 - Two models architecture

As shown in `0 - ADSR Prediction`, single model for ADSR + Note prediction is not quite good. This is an attempt at making two different models work together to predict the note and ADSR separately.

In [1]:
import sys
sys.path.insert(0, "..")

import os
os.environ["PYTORCH_HIP_ALLOC_CONF"] = "garbage_collection_threshold:0.8"

from pathlib import Path
from datetime import datetime

from tqdm import tqdm
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import mlflow

from rustic_ml.encoding import encode_adsr, decode_adsr, NOTE_MIN, NOTE_MAX, N_NOTES
from rustic_ml.dataset import random_spec, render_mel, NpzDataset, prepare_dataloaders
from rustic_ml.analysis import MetricSpec, analyze_run
from rustic_ml.evaluation import (
    accumulate_note_inference, accumulate_adsr_inference, accumulate_pipeline_inference,
    plot_note_accuracy, plot_adsr_accuracy, plot_accuracy,
    compare_audio_dual,
)
from rustic_ml.mlflow_ui import show_register_widget, show_describe_widget, show_registered_models, setup_mlflow
from rustic_ml.training import set_seeds, log_hyperparams, setup_device
from rustic_ml.comparison import compare_note_models, compare_adsr_models, load_registered_model

In [2]:
# MlFlow
MLFLOW_URI = "http://192.168.1.254:5000"
MLFLOW_EXPERIMENT = "DualModel"

# Dataset
DATA_DIR = Path("/data/datasets")

# Wether to train the note model or not
TRAIN_NOTE = False

# Training parameters
HYPER = {
    "seed":           42,
    "n_samples":      80_000,
    "batch_size_gen": 1_000,
    "batch_size":     64,
    "n_epochs":       80,
    "lr":             1e-3,
}
set_seeds(HYPER["seed"])

In [3]:
device = setup_device()
setup_mlflow(MLFLOW_URI, MLFLOW_EXPERIMENT)

CUDA detected — using NVIDIA GeForce GTX 1050
MLflow connected: http://192.168.1.254:5000  (experiment: DualModel)


In [4]:
print(f"Using torch {torch.__version__} device {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"HIP version: {torch.version.hip}")
print(f"Using mlflow {mlflow.__version__}")
print(f"Rustic_ML OK - notes [{NOTE_MIN}, {NOTE_MAX}] N_NOTES={N_NOTES}")

Using torch 2.5.1+cu121 device NVIDIA GeForce GTX 1050
HIP version: None
Using mlflow 3.10.1
Rustic_ML OK - notes [36, 84] N_NOTES=49


In [5]:
train_loader, val_loader, train_ds, val_ds = prepare_dataloaders(
    DATA_DIR, HYPER["n_samples"], HYPER["batch_size_gen"],
    HYPER["batch_size"], seed=HYPER["seed"],
)

Train: 64000 samples (64 files)  |  Val: 16000 samples (16 files)


## Model instanciation

In [6]:
from rustic_ml.models import NotePredictor, ADSRPredictor

In [7]:
# Re-running this cell resets the ADSR model to random weights.
# To continue ADSR training from the current state, re-run only the ADSR training cell below.
model_adsr = ADSRPredictor().to(device)
criterion_adsr = nn.MSELoss()
optimizer_adsr = torch.optim.Adam(model_adsr.parameters(), lr=HYPER["lr"])

criterion_note = nn.CrossEntropyLoss()

if torch.cuda.is_available():
    print("Models moved to device")
    print(torch.cuda.memory_allocated() / 1e6, "MB allocated")
    print(torch.cuda.memory_reserved() / 1e6, "MB reserved")

ADSRPredictor - 24,628 trainable parameters
Models moved to device
0.108032 MB allocated
2.097152 MB reserved


In [ ]:
def train_model(model, optimizer, train_loader, val_loader, device, hyper,
                train_step, eval_step, log_fn, post_training=None, run_name: str | None = None,
                model_name: str = "model") -> str:
    """
    train_step(model, batch, device) -> loss tensor  (backward called by loop)                      
    eval_step(model, batch, device)  -> dict[str, float]  (named scalars, summed per epoch)         
    log_fn(train_loss, train_n, val_totals, val_n, epoch) -> None                                   
    post_training(model) -> None  (optional, runs inside the mlflow run)                            
    """
    with mlflow.start_run(run_name=run_name) as run:
        log_hyperparams(hyper)
    
        for epoch in tqdm(range(hyper["n_epochs"]), desc=f"Epoch"):
            # Train
            model.train()
            train_total = 0.0
            for batch in train_loader:    
                optimizer.zero_grad()
                loss = train_step(model, batch, device)
                loss.backward()
                optimizer.step()
                train_total += loss.item()
    
            # Eval
            model.eval()
            val_totals: dict[str, float] = {}
            with torch.no_grad():
                for batch in val_loader:
                    for k, v in eval_step(model, batch, device).items():
                        val_totals[k] = val_totals.get(k, 0.0) + v
    
            log_fn(
                train_total, len(train_loader),
                val_totals, len(val_loader),
                epoch,
            )
    
        if post_training:
            post_training(model)
        # Log the model
        mlflow.pytorch.log_model(model, model_name)
    return run.info.run_id

## Note model

Run **one** of the two cells below — either load the registered model or train a new one.

### Option A — load latest registered NotePredictor

In [ ]:
RUN_ID_NOTE = None  # no new MLflow run — note model was not retrained (yet?)

if not TRAIN_NOTE:
    # Override model with latest from mlflow
    model_note = load_registered_model("NotePredictor", device)

### Option B — train a new NotePredictor from scratch

In [ ]:
if TRAIN_NOTE:
    model_note = NotePredictor().to(device)
    optimizer_note = torch.optim.Adam(model_note.parameters(), lr=HYPER["lr"])

def note_train_step(model, batch, device):
    mel, note, _ = batch
    mel, note = mel.to(device), note.to(device)
    return criterion_note(model(mel), note - NOTE_MIN)

def note_eval_step(model, batch, device):
    mel, note, _ = batch
    mel, note = mel.to(device), note.to(device)
    logits = model(mel)
    return {
        "loss":    criterion_note(logits, note - NOTE_MIN).item(),
        "correct": (logits.argmax(1) == note - NOTE_MIN).sum().item(),
        "total":   note.size(0),
    }

def note_log_fn(train_loss, train_n, val_totals, val_n, epoch):
    mlflow.log_metrics({
        "train/loss_note":   train_loss / train_n,
        "val/loss_note":     val_totals["loss"] / val_n,
        "val/accuracy_note": val_totals["correct"] / val_totals["total"],
    }, step=epoch)

In [ ]:
if TRAIN_NOTE:
    RUN_ID_NOTE = train_model(
        model_note, optimizer_note, train_loader, val_loader, device, HYPER,
        train_step=note_train_step,
        eval_step=note_eval_step,
        log_fn=note_log_fn,
        post_training=lambda m: plot_note_accuracy(
            accumulate_note_inference(m, val_loader, device)
        ),
        run_name=f"train_note_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        model_name="note_discriminator",
    )

## ADSR Training

In [ ]:
def adsr_train_step(model, batch, device):
    mel, note, adsr = batch
    mel, note, adsr = mel.to(device), note.to(device), adsr.to(device)
    return criterion_adsr(model(mel, note), adsr)

def adsr_eval_step(model, batch, device):
    mel, note, adsr = batch
    mel, note, adsr = mel.to(device), note.to(device), adsr.to(device)
    return {"loss": criterion_adsr(model(mel, note), adsr).item()}

def adsr_log_fn(train_loss, train_n, val_totals, val_n, epoch):
    mlflow.log_metrics({
        "train/loss_adsr": train_loss / train_n,
        "val/loss_adsr": val_totals["loss"] / val_n,
    }, step=epoch)

In [ ]:
RUN_ID_ADSR = train_model(
    model_adsr, optimizer_adsr, train_loader, val_loader, device, HYPER,
    train_step=adsr_train_step,
    eval_step=adsr_eval_step,
    log_fn=adsr_log_fn,
    post_training=lambda m: (
        plot_adsr_accuracy(accumulate_adsr_inference(m, val_loader, device)),
    ),
    run_name=f"train_adsr_{datetime.now().strftime("%Y%m%d_%H%M%S")}",
    model_name="adsr_predictor"
)

## Training analysis

In [ ]:
if RUN_ID_NOTE:
    analyze_run([
        MetricSpec("train/loss_note",   "train", "loss_note"),
        MetricSpec("val/loss_note",     "val",   "loss_note"),
        MetricSpec("val/accuracy_note", "val",   "accuracy_note"),
    ], run_id=RUN_ID_NOTE, tracking_uri=MLFLOW_URI)

In [ ]:
analyze_run([
    MetricSpec("train/loss_adsr", "train", "loss_adsr"),
    MetricSpec("val/loss_adsr",   "val",   "loss_adsr"),
], run_id=RUN_ID_ADSR, tracking_uri=MLFLOW_URI)

## Pipeline evaluation

In [ ]:
# End-to-end: note_model → adsr_model
vals = accumulate_pipeline_inference(model_note, model_adsr, val_loader, device)
plot_accuracy(vals)

In [ ]:
compare_audio_dual(model_note, model_adsr, val_ds, device)

## Comparison against registry

In [ ]:
compare_note_models("NotePredictor", model_note, val_loader, device)

In [ ]:
compare_adsr_models(
    "ADSRPredictor", model_adsr, val_loader, device,
    note_source=model_note,  # use predicted notes — same source for both models (fair comparison)
)

## Register models

In [ ]:
if RUN_ID_NOTE:
    show_register_widget(RUN_ID_NOTE, model_artifact="note_discriminator", default_name="NotePredictor")
    show_describe_widget(RUN_ID_NOTE)

In [ ]:
show_register_widget(RUN_ID_ADSR, model_artifact="adsr_predictor", default_name="ADSRPredictor")
show_describe_widget(RUN_ID_ADSR)